# Simulatie thuisbatterij PV-klanten

Deze notebook berekent voor elke ZP-klant de jaarlijkse factuur en besparing onder zeven scenario's, en evalueert of een thuisbatterij financieel rendabel is via de netto contante waarde (NCW).

**Zeven scenario's:**

| | Contract | Batterijstrategie |
|---|---|---|
| **A** | Vast | Geen batterij (referentie) |
| **B** | Dynamisch | Geen batterij |
| **C** | Vast | Day-ahead pieksturing |
| **D** | Dynamisch | Day-ahead prijsarbitrage |
| **E** | Dynamisch | Day-ahead pieksturing |
| **F** | Vast | SCM zelfconsumptie (realistisch) |
| **G** | Dynamisch | SCM zelfconsumptie (realistisch) |

**Inputbestanden:**

- `fluvius_300_met_ZP_uur.csv` en drie andere ZP-profielbestanden (output van tariefspiraal-notebook, stap 2)
- `Belgium_2024_MWh.csv` (output van tariefspiraal-notebook, stap 1)

**Outputbestand:** `besparing_pv_batterij.csv`

## Configuratie

Alle tariefparameters, batterijspecificaties en NCW-aannames staan in deze cel. Bij wijzigingen volstaat het deze cel opnieuw te draaien.

In [ ]:
import numpy as np
import pandas as pd

# ── Bestanden ────────────────────────────────────────────────
FLUVIUS_BESTANDEN = [
    "fluvius_300_met_ZP_uur.csv",
    "fluvius_300_met_EV_met_ZP_uur.csv",
    "fluvius_300_met_WP_met_ZP_uur.csv",
    "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
]
EMBER_BESTAND  = "Belgium_2024_MWh.csv"
OUTPUT_BESTAND = "besparing_pv_batterij.csv"

# ── Vast contract (portfoliomethode, consistent met tariefspiraal-notebook) ──
# Berekend op de Vlaamse baseline 2024, incl. 6% btw op afname
VAST_AFNAME_CT   = 8.33    # ct/kWh, incl. btw
VAST_INJECTIE_CT = 4.02    # ct/kWh, excl. btw

# ── Dynamisch contract (Eneco Zon & Wind Dynamisch, januari 2024) ────────────
BTW           = 1.06
DYN_ALPHA_AFN = 0.102
DYN_BETA_AFN  = 1.0
DYN_ALPHA_INJ = 0.100
DYN_BETA_INJ  = 1.188   # injectievergoeding kan negatief worden bij EPEX < 11,88 EUR/MWh

# ── Netcomponent (identiek voor beide contracten) ────────────────────────────
# Rekenkundig gemiddelde over de 10 Vlaamse Fluvius-distributienetzones
NET_VOL_CT      = 4.05      # ct/kWh, volumetrisch distributietarief
CAP_TARIEF      = 44.00     # €/kW/jaar, capaciteitstarief
CAP_MIN_KW      = 2.5       # contractueel minimum
DATABEHEER_SMR1 = 13.39     # €/jaar, vast contract
DATABEHEER_SMR3 = 14.53     # €/jaar, dynamisch contract

# ── Heffingen (overheid, identiek voor beide contracten) ─────────────────────
HEFFINGEN_CT = 5.0329 + 0.2042 + 1.5600   # bijzondere accijns + energiebijdrage + groene stroom

# ── Batterij (Alpha-ESS Smile G3) ────────────────────────────────────────────
BAT_CAP_KWH = 9.3           # kWh nominale capaciteit
BAT_MAX_KW  = 5.0           # kW maximaal laad-/ontlaadvermogen
BAT_EFF     = 0.92          # round-trip efficiëntie

# ── NCW-parameters ───────────────────────────────────────────────────────────
NCW_I0 = 3606.41            # aanschafprijs incl. btw (Alpha-ESS Smile G3)
NCW_R  = 0.035              # discontovoet (Belgische lineaire obligaties 10j)
NCW_T  = 10                 # tijdshorizon (garantieperiode batterij)

# ── Leesbare profielnamen ────────────────────────────────────────────────────
PROFIEL_LABELS = {
    "met_ZP":               "ZP",
    "met_EV_met_ZP":        "ZP + EV",
    "met_WP_met_ZP":        "ZP + WP",
    "met_WP_met_EV_met_ZP": "ZP + WP + EV",
}
PROFIEL_VOLGORDE = ["ZP", "ZP + EV", "ZP + WP", "ZP + WP + EV"]

## Stap 1 — Data inladen en koppelen

De Fluvius-uurdata wordt vanuit UTC omgezet naar Belgische lokale tijd voor de koppeling met de Ember-prijzen op maand/dag/uur. Wintertijdovergangen (dubbele uren) worden gemiddeld; de zomertijdovergang (27 maart 02:00) wordt verwijderd uit beide datasets.

In [ ]:
def laad_data():
    """Laad de vier ZP-profielbestanden en koppel uurlijkse EPEX-prijzen."""
    dfs = []
    for bestand in FLUVIUS_BESTANDEN:
        df = pd.read_csv(bestand)
        df["profiel"] = bestand.replace("fluvius_300_", "").replace("_uur.csv", "")
        dfs.append(df)
    fluvius = pd.concat(dfs, ignore_index=True).rename(columns={
        "EAN_ID":              "klant_id",
        "Datum_Startuur":      "timestamp",
        "Volume_Afname_KWh":   "afname_kwh",
        "Volume_Injectie_KWh": "injectie_kwh",
    })
    fluvius["timestamp"]    = pd.to_datetime(fluvius["timestamp"], utc=True)
    fluvius["afname_kwh"]   = pd.to_numeric(fluvius["afname_kwh"],
                                             errors="coerce").fillna(0).clip(lower=0)
    fluvius["injectie_kwh"] = pd.to_numeric(fluvius["injectie_kwh"],
                                             errors="coerce").fillna(0).clip(lower=0)

    # Laad Ember-prijzen; wintertijd: dubbele uren middelen
    ember = pd.read_csv(EMBER_BESTAND).rename(columns={
        "Datetime (Local)": "timestamp",
        "Price (EUR/MWh)":  "prijs_eur_mwh",
    })
    ember["timestamp"] = pd.to_datetime(ember["timestamp"])
    ember["maand"] = ember["timestamp"].dt.month
    ember["dag"]   = ember["timestamp"].dt.day
    ember["uur"]   = ember["timestamp"].dt.hour
    ember = ember.groupby(["maand", "dag", "uur"]).agg(
        prijs_eur_mwh=("prijs_eur_mwh", "mean"),
    ).reset_index()

    # UTC → Belgische lokale tijd; zomertijdovergang (NaT) wegfilteren
    fluvius["timestamp_lokaal"] = pd.to_datetime(
        fluvius["timestamp"], utc=True
    ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)
    fluvius["maand"] = fluvius["timestamp_lokaal"].dt.month
    fluvius["dag"]   = fluvius["timestamp_lokaal"].dt.day
    fluvius["uur"]   = fluvius["timestamp_lokaal"].dt.hour
    fluvius = fluvius.dropna(subset=["timestamp_lokaal"]).copy()

    # Koppel op maand/dag/uur in lokale tijd
    data = fluvius.merge(ember[["maand", "dag", "uur", "prijs_eur_mwh"]],
                         on=["maand", "dag", "uur"], how="left")
    data["prijs_eur_mwh"] = data["prijs_eur_mwh"].fillna(data["prijs_eur_mwh"].median())
    data["profiel_label"] = data["profiel"].map(PROFIEL_LABELS).fillna(data["profiel"])

    print(f"Gekoppeld: {data['klant_id'].nunique()} klanten | {len(data):,} rijen")
    return data


def bereken_facturatiepiek(data):
    """
    Bereken het capaciteitstarief per klant op basis van de gemiddelde
    maandelijkse piekafname over de twaalf kalendermaanden, met
    contractueel minimum van 2,5 kW.
    """
    df = data[["klant_id", "timestamp", "afname_kwh"]].copy()
    df["maand"] = df["timestamp"].dt.strftime("%Y-%m")
    maandpiek   = df.groupby(["klant_id", "maand"])["afname_kwh"].max()
    return maandpiek.groupby("klant_id").mean().clip(lower=CAP_MIN_KW).to_dict()

## Stap 2 — Factuurberekening

Twee functies berekenen de volledige jaarfactuur per klant:

- **`bereken_factuur_vast`** — energie via vast tarief + netcomponent + capaciteitstarief + heffingen + databeheer
- **`bereken_factuur_dynamisch`** — energie via Eneco-formule + zelfde netcomponent en heffingen


In [ ]:
def bereken_factuur_vast(afname, injectie, piek):
    """Jaarfactuur op vast contract: energie + net + capaciteit + heffingen + databeheer."""
    tot_afn   = afname.sum()
    tot_inj   = injectie.sum()
    energie   = (tot_afn * VAST_AFNAME_CT - tot_inj * VAST_INJECTIE_CT) / 100
    net_vol   = tot_afn * NET_VOL_CT / 100
    cap       = piek * CAP_TARIEF
    heffingen = tot_afn * HEFFINGEN_CT / 100
    totaal    = energie + net_vol + cap + heffingen + DATABEHEER_SMR1
    return {"totaal": totaal, "energie": energie, "net_vol": net_vol,
            "cap": cap, "heffingen": heffingen, "databeheer": DATABEHEER_SMR1}


def bereken_factuur_dynamisch(afname, injectie, prijs_mwh, piek):
    """
    Jaarfactuur op Eneco Zon & Wind Dynamisch contract (januari 2024).

    Afname per uur:    (0,102 × EPEX + 1,0) × 1,06 / 100  €/kWh (incl. btw)
    Injectie per uur:  (0,100 × EPEX − 1,188) / 100       €/kWh (kan negatief)
    """
    tarief_afn = (DYN_ALPHA_AFN * prijs_mwh + DYN_BETA_AFN) * BTW
    tarief_inj = DYN_ALPHA_INJ * prijs_mwh - DYN_BETA_INJ
    energie    = ((afname * tarief_afn).sum() - (injectie * tarief_inj).sum()) / 100
    tot_afn    = afname.sum()
    net_vol    = tot_afn * NET_VOL_CT / 100
    cap        = piek * CAP_TARIEF
    heffingen  = tot_afn * HEFFINGEN_CT / 100
    totaal     = energie + net_vol + cap + heffingen + DATABEHEER_SMR3
    return {"totaal": totaal, "energie": energie, "net_vol": net_vol,
            "cap": cap, "heffingen": heffingen, "databeheer": DATABEHEER_SMR3}

## Stap 3 — Batterijstrategieën

Twee soorten strategieën:

**SCM (Self-Consumption Maximization)** — realistisch
- Laadt op PV-overschot, ontlaadt bij netafname
- Chronologisch over het hele jaar, geen vooruitkijk
- Lading kan van dag naar dag worden meegenomen

**Day-ahead foresight** — theoretische bovengrens
- PV-overschot van dag D wordt opgeslagen, gebruikt op dag D+1
- Ontlaadt op de uren met de hoogste prijs (*prijsarbitrage*) of hoogste afname (*pieksturing*)
- Vereist perfecte kennis van de prijzen en verbruiken van de volgende dag

In beide gevallen wordt de round-trip efficiëntie (92%) toegepast bij het ontladen.

In [ ]:
def simuleer_batterij_scm(klant_data):
    """
    Self-Consumption Maximization — realistische strategie zonder vooruitkijk.
    Laadt op PV-overschot, ontlaadt bij netafname; lading wordt over dagen meegenomen.
    """
    df = klant_data.sort_values("timestamp").copy()
    afname_na   = df["afname_kwh"].values.copy().astype(float)
    injectie_na = df["injectie_kwh"].values.copy().astype(float)
    opslag      = 0.0

    for t in range(len(df)):
        # Laden op PV-overschot (geen efficiëntieverlies bij laden)
        if injectie_na[t] > 0:
            laden = max(0.0, min(injectie_na[t], BAT_MAX_KW, BAT_CAP_KWH - opslag))
            injectie_na[t] = max(0.0, injectie_na[t] - laden)
            opslag += laden

        # Ontladen bij netafname (efficiëntieverlies bij ontladen)
        if afname_na[t] > 0 and opslag > 0:
            ontladen_uit_bat  = max(0.0, min(afname_na[t] / BAT_EFF, BAT_MAX_KW, opslag))
            geleverd_aan_huis = ontladen_uit_bat * BAT_EFF
            afname_na[t] = max(0.0, afname_na[t] - geleverd_aan_huis)
            opslag -= ontladen_uit_bat

    return afname_na, injectie_na


def simuleer_batterij_foresight(klant_data, strategie="prijsarbitrage"):
    """
    Day-ahead foresight strategie — theoretische bovengrens.

    PV-overschot van dag D wordt opgeslagen en op dag D+1 ontladen op:
      - de duurste uren (strategie = "prijsarbitrage")
      - de uren met hoogste afname (strategie = "pieksturing")
    """
    df = klant_data.sort_values("timestamp").copy()
    df["datum"] = df["timestamp"].dt.strftime("%Y-%m-%d")
    afname_na   = df["afname_kwh"].values.copy().astype(float)
    injectie_na = df["injectie_kwh"].values.copy().astype(float)

    opgeslagen = 0.0

    for dag in sorted(df["datum"].unique()):
        idx   = np.where(df["datum"].values == dag)[0]
        prijs = df["prijs_eur_mwh"].values[idx]
        afn   = df["afname_kwh"].values[idx]
        inj   = df["injectie_kwh"].values[idx]

        # Ontladen: gebruik de lading van gisteren op de uren met meeste voordeel
        if opgeslagen > 0:
            if strategie == "prijsarbitrage":
                volgorde = np.argsort(prijs)[::-1]   # duurste uren eerst
            else:  # pieksturing
                volgorde = np.argsort(afn)[::-1]     # hoogste afname eerst

            soc = opgeslagen
            for t in volgorde:
                if soc <= 0:
                    break
                ontladen_uit_bat  = min(BAT_MAX_KW, soc, max(0.0, afn[t]) / BAT_EFF)
                geleverd_aan_huis = ontladen_uit_bat * BAT_EFF
                afname_na[idx[t]] = max(0.0, afn[t] - geleverd_aan_huis)
                soc -= ontladen_uit_bat

        # Laden: sla PV-overschot van vandaag op (begrensd door capaciteit en vermogen)
        pv_overschot = 0.0
        opslag_tmp   = 0.0
        for t in range(len(idx)):
            if inj[t] > 0:
                laden = max(0.0, min(inj[t], BAT_MAX_KW, BAT_CAP_KWH - opslag_tmp))
                opslag_tmp   += laden
                pv_overschot += laden
        opgeslagen = opslag_tmp

        # Verminder de injectie proportioneel met de opgeslagen fractie
        if pv_overschot > 0 and opgeslagen > 0:
            fractie = opgeslagen / pv_overschot
            for t in range(len(idx)):
                if inj[t] > 0:
                    injectie_na[idx[t]] = max(0.0, inj[t] * (1 - fractie))

    return afname_na, injectie_na


def bereken_piek_na_batterij(afname_na, maand):
    """Bereken de gemiddelde maandelijkse piekafname na batterijinzet (min. 2,5 kW)."""
    df_piek = pd.DataFrame({"afname": afname_na, "maand": maand})
    piek    = df_piek.groupby("maand")["afname"].max().mean(skipna=True)
    return max(piek, CAP_MIN_KW)

## Stap 4 — Hoofdsimulatie

Loopt over alle 1.200 ZP-klanten en berekent voor elk de jaarfactuur onder de zeven scenario's. De NCW wordt berekend met de annuïteitsformule:

$$\text{NCW} = -I_0 + B \times \frac{(1+r)^t - 1}{r \times (1+r)^t}$$

Met I_0 = €3.606,41, $r = 3,5\%$, $t = 10$ jaar, en $B$ = jaarlijkse besparing t.o.v. scenario A.

In [ ]:
print("="*70)
print("SIMULATIE THUISBATTERIJ PV-KLANTEN")
print("="*70)
print(f"Batterij: {BAT_CAP_KWH} kWh / {BAT_MAX_KW} kW / {BAT_EFF*100:.0f}% eff.")
print(f"NCW: I0=€{NCW_I0:,.2f} | r={NCW_R*100:.1f}% | t={NCW_T} jaar")

# Stap 4a — Data inladen
print("\n1. Data inladen...")
data        = laad_data()
profiel_map = data.groupby("klant_id")["profiel"].first().to_dict()
klanten     = data["klant_id"].unique()

# Stap 4b — Facturatiepiek zonder batterij
print("\n2. Facturatiepieken berekenen...")
pieken = bereken_facturatiepiek(data)
print(f"  Gemiddelde facturatiepiek: {sum(pieken.values())/len(pieken):.2f} kW")

# Stap 4c — Simulatie per klant
print(f"\n3. Simulatie per klant ({len(klanten)} klanten)...")
resultaten = []

for i, klant_id in enumerate(klanten):
    if i % 200 == 0:
        print(f"   {i+1}/{len(klanten)}...")

    kd    = data[data["klant_id"] == klant_id].copy()
    afn   = kd["afname_kwh"].values
    inj   = kd["injectie_kwh"].values
    prijs = kd["prijs_eur_mwh"].values
    maand = kd["timestamp"].dt.strftime("%Y-%m").values
    piek  = pieken.get(klant_id, CAP_MIN_KW)

    # Zeven scenario's
    fA = bereken_factuur_vast(afn, inj, piek)                                  # vast, geen batterij
    fB = bereken_factuur_dynamisch(afn, inj, prijs, piek)                      # dyn,  geen batterij

    afn_C, inj_C = simuleer_batterij_foresight(kd, strategie="pieksturing")
    piek_C = bereken_piek_na_batterij(afn_C, maand)
    fC = bereken_factuur_vast(afn_C, inj_C, piek_C)                            # vast, foresight pieksturing

    afn_D, inj_D = simuleer_batterij_foresight(kd, strategie="prijsarbitrage")
    piek_D = bereken_piek_na_batterij(afn_D, maand)
    fD = bereken_factuur_dynamisch(afn_D, inj_D, prijs, piek_D)                # dyn,  foresight prijsarb.

    afn_E, inj_E = simuleer_batterij_foresight(kd, strategie="pieksturing")
    piek_E = bereken_piek_na_batterij(afn_E, maand)
    fE = bereken_factuur_dynamisch(afn_E, inj_E, prijs, piek_E)                # dyn,  foresight pieksturing

    afn_F, inj_F = simuleer_batterij_scm(kd)
    piek_F = bereken_piek_na_batterij(afn_F, maand)
    fF = bereken_factuur_vast(afn_F, inj_F, piek_F)                            # vast, SCM
    fG = bereken_factuur_dynamisch(afn_F, inj_F, prijs, piek_F)                # dyn,  SCM

    resultaten.append({
        "klant_id":      klant_id,
        "profiel":       profiel_map.get(klant_id, "onbekend"),
        "profiel_label": PROFIEL_LABELS.get(profiel_map.get(klant_id, ""), "onbekend"),
        "factuur_A": round(fA["totaal"], 2), "factuur_B": round(fB["totaal"], 2),
        "factuur_C": round(fC["totaal"], 2), "factuur_D": round(fD["totaal"], 2),
        "factuur_E": round(fE["totaal"], 2), "factuur_F": round(fF["totaal"], 2),
        "factuur_G": round(fG["totaal"], 2),
        "besp_B": round(fA["totaal"] - fB["totaal"], 2),
        "besp_C": round(fA["totaal"] - fC["totaal"], 2),
        "besp_D": round(fA["totaal"] - fD["totaal"], 2),
        "besp_E": round(fA["totaal"] - fE["totaal"], 2),
        "besp_F": round(fA["totaal"] - fF["totaal"], 2),
        "besp_G": round(fA["totaal"] - fG["totaal"], 2),
        "piek_A": round(piek,   2), "piek_C": round(piek_C, 2),
        "piek_D": round(piek_D, 2), "piek_E": round(piek_E, 2),
        "piek_F": round(piek_F, 2),
        "afname_kwh":   round(afn.sum(), 2),
        "injectie_kwh": round(inj.sum(), 2),
    })

res = pd.DataFrame(resultaten)

# Stap 4d — NCW berekening per scenario
annuiteitsfactor = ((1 + NCW_R)**NCW_T - 1) / (NCW_R * (1 + NCW_R)**NCW_T)
for scenario in ["C", "D", "E", "F", "G"]:
    res[f"ncw_{scenario}"] = (-NCW_I0 + res[f"besp_{scenario}"] * annuiteitsfactor).round(2)

print(f"\nAnnuïteitsfactor: {annuiteitsfactor:.4f}")
print(f"Break-even besparing: €{NCW_I0/annuiteitsfactor:.0f}/jaar")

# Stap 4e — Overzicht resultaten per profieltype
print("\n" + "="*70)
print("RESULTATEN PER PROFIELTYPE")
print("="*70)
for profiel in PROFIEL_VOLGORDE:
    groep = res[res["profiel_label"] == profiel]
    if len(groep) == 0:
        continue
    print(f"\n── {profiel} ({len(groep)} klanten) ──")
    for kolom, naam in [("factuur_A","A vast/geen bat"), ("factuur_B","B dyn/geen bat"),
                          ("factuur_C","C vast/pieksturing"), ("factuur_D","D dyn/prijsarb."),
                          ("factuur_E","E dyn/pieksturing"), ("factuur_F","F vast/SCM"),
                          ("factuur_G","G dyn/SCM")]:
        print(f"  [{naam:<22}] €{groep[kolom].mean():>7.2f}/jaar")
    print("  ──")
    for kolom, naam in [("besp_F","Besparing F vs A"), ("ncw_F","NCW F"),
                          ("ncw_G","NCW G")]:
        gem     = groep[kolom].mean()
        pct_pos = (groep[kolom] > 0).mean() * 100
        eenheid = "€/jaar" if "besp" in kolom else "€"
        print(f"  {naam:<25} {gem:>7.0f} {eenheid:<6} ({pct_pos:.0f}% klanten positief)")

res.to_csv(OUTPUT_BESTAND, index=False)
print(f"\n✓ Opgeslagen: {OUTPUT_BESTAND}")